In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\1. MLP\\Func_apxm_MLP_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\1. MLP\Func_apxm_MLP_pytorch


# (1) Import libraries

In [2]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import numpy as np
from scipy.stats import multivariate_normal

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import time
from IPython.display import clear_output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# (2) Data preparation

In [3]:
# training ------------------------------
A = 32j
x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x1x2 = np.column_stack([x1.flat, x2.flat])

mu = np.array([0.0, 0.0])
covariance = np.diag([1, 1])

y = multivariate_normal.pdf(x1x2, mean=mu, cov=covariance)
y = y.reshape(x1.shape)

x_train = x1x2.astype(np.float32)
y_train = y.flatten().astype(np.float32)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(x1, x2, y)
plt.title('training data')
plt.show()

# testing -------------------------------
A = 10j
x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x1x2 = np.column_stack([x1.flat, x2.flat])

y = multivariate_normal.pdf(x1x2, mean=mu, cov=covariance)
y = y.reshape(x1.shape)

x_test = x1x2.astype(np.float32)
y_test = y.flatten().astype(np.float32)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(x1, x2, y)
plt.title('testing data')
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14296\1734227515.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


# (3) MLP (PyTorch)

In [4]:
class GaussMLP(nn.Module):
    def __init__(self, B=16):
        super().__init__()
        # 2 input nodes -> B hidden nodes (sigmoid) -> 1 output node (sigmoid)
        self.fc = nn.Linear(2, B)
        self.gauss_apxm = nn.Linear(B, 1)
        # Khởi tạo Xavier/Glorot giống mặc định của Keras Dense
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)
        nn.init.xavier_uniform_(self.gauss_apxm.weight)
        nn.init.zeros_(self.gauss_apxm.bias)

    def forward(self, x):
        x = torch.sigmoid(self.fc(x))
        out = torch.sigmoid(self.gauss_apxm(x))
        return out


def train_and_evaluate(x_tr, y_tr, x_full_tr, y_full_tr, x_te, y_te, B=16, epochs=100, batch_size=128, lr=0.01):
    model = GaussMLP(B=B).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    train_ds = TensorDataset(
        torch.from_numpy(x_tr),
        torch.from_numpy(y_tr).unsqueeze(1)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            preds = model(bx)
            loss = criterion(preds, by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        tr_preds = model(torch.from_numpy(x_full_tr).to(device))
        tr_loss = criterion(tr_preds, torch.from_numpy(y_full_tr).unsqueeze(1).to(device)).item()

        te_preds = model(torch.from_numpy(x_te).to(device))
        te_loss = criterion(te_preds, torch.from_numpy(y_te).unsqueeze(1).to(device)).item()

    return tr_loss, te_loss

# (4) Training and evaluating MLP

### (4.1) Number of training samples (A) (no. of hidden nodes (B) = 16 and no. of epochs (C) = 100)

In [5]:
train_error = []
test_error = []
tot_train = 1001
A_array = []

B = 16
C = 100
for A in np.arange(100, tot_train, 100):
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    tr_loss, te_loss = train_and_evaluate(
        x_train[sel_indx], y_train[sel_indx],
        x_train, y_train, x_test, y_test,
        B=B, epochs=C, batch_size=128, lr=0.01
    )

    train_error.append(tr_loss)
    test_error.append(te_loss)
    A_array.append(A)

    clear_output(wait=True)
    plt.plot(A_array, train_error, label='training')
    plt.plot(A_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of training samples (A)')
    plt.xlabel('#-samples')
    plt.ylabel('Error')
    plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14296\1734227515.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


### (4.2) Number of hidden nodes (B) (no. of training samples (A) = 800 and no. of epochs (C) = 100)

In [6]:
train_error = []
test_error = []
B_array = []
tot_train = 1001

A = 800
B = 8
C = 100
while B <= 512:
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    tr_loss, te_loss = train_and_evaluate(
        x_train[sel_indx], y_train[sel_indx],
        x_train, y_train, x_test, y_test,
        B=B, epochs=C, batch_size=128, lr=0.01
    )

    train_error.append(tr_loss)
    test_error.append(te_loss)
    B_array.append(B)

    clear_output(wait=True)
    plt.plot(B_array, train_error, label='training')
    plt.plot(B_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of hidden nodes (B)')
    plt.xlabel('Number of hidden nodes')
    plt.ylabel('Error')
    plt.show()

    B *= 2

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14296\1734227515.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)


### (4.3) Number of epochs (C) (no. of training samples (A) = 800 and no. of hidden nodes (B) = 128)

In [7]:
train_error = []
test_error = []
tot_train = 1001
C_array = []

A = 800
B = 128
for C in np.arange(10, 200, 10):
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    tr_loss, te_loss = train_and_evaluate(
        x_train[sel_indx], y_train[sel_indx],
        x_train, y_train, x_test, y_test,
        B=B, epochs=C, batch_size=128, lr=0.01
    )

    train_error.append(tr_loss)
    test_error.append(te_loss)
    C_array.append(C)

    clear_output(wait=True)
    plt.plot(C_array, train_error, label='training')
    plt.plot(C_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of epochs (C)')
    plt.xlabel('Epochs')
    plt.ylabel('Error')
    plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14296\1734227515.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
